# Inverse potential problem 

The inverse potential problem seeks the shape of a homogeneous source from measurements on a circle enclosing it. The operator maps the shape of the heat source to the heat flux measured on that circle. The heat distribution satisfies

$$
        \begin{cases}
            \Delta u = 1_K & \text{ in } \Omega \\
            u = 0          & \text{ on } \partial\Omega
        \end{cases}
$$

where $\partial\Omega$ is the measurement circle and $K$ is the heat source. The forward operator
maps the shape of the heat source to the Neumann data:

$$
        \partial K \mapsto \frac{\partial u}{\partial\nu}|_{\partial\Omega}.
$$

References:
- F. Hettlich & W. Rundell "Iterative methods for the reconstruction of an inverse potential problem", Inverse Problems, 12 (1996) 251–266.
- T. Hohage "Logarithmic convergence rates of the iteratively regularized Gauss–Newton method for an inverse potential and an inverse scattering problem", Inverse Problems, 13 (1997) 1279–1299.

## Import auxiliary libraries and define logging

We use `numpy` arrays as the main data structure for the domain and rely on `matplotlib.pyplot` as the tool for plots. 

To observe the information of the solver while computing we define the logging string and set the log level to info.

In [ ]:
import logging
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import sys


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-40s :: %(message)s'
)


example_dir = "../../../../examples/potential/"
Path(example_dir).resolve()
sys.path.insert(0, str(example_dir))
from potential import Potential

## Define operator and regularization setting

The regularization setting uses a Sobolev penalty and an $L^2$ data-fidelity term. Here the abstract Sobolev space defaults to smoothness index 1.

For the exact solution, we define a star-shaped domain with boundary $ \sqrt{6\cos(1.5t)^2+1}/3$. We then compute exact data and add noise with a relative level of 3%.

In [ ]:
REL_NOISE_LEVEL = 0.03
RADIUS_INITIAL_GUESS = 0.2
RADIUS_MEASUREMENTS = 1.3

from regpy.solvers import Setting
from regpy.hilbert import L2, Sobolev

# Forward operator
op = Potential(radius=RADIUS_MEASUREMENTS)
# Exact solution 
exact_solution = op.domain.radialfct2curve(lambda t: np.sqrt(6*np.cos(1.5*t)**2+1)/3)
# Exact data
exact_data = op(exact_solution.radial())
#Initial guess
init = op.domain.circle(radius=RADIUS_INITIAL_GUESS)

sett = Setting(op=op, penalty=Sobolev, data_fid=L2,
                  exact_data=exact_data, init = init.coeff)
sett.add_Gaussian_noise(relative_noise_level=REL_NOISE_LEVEL,white_noise=False)


## Plot the configuration   


In [ ]:
def close_curve(z):
    closed_z = np.concatenate((z, z[:, :1]), axis=1)
    return closed_z[0,:], closed_z[1,:]

fig = plt.figure(figsize=(7,7))
plt.plot(*close_curve(exact_solution.z),label = 'exact solution')
plt.plot(*close_curve(init.z),label='initial guess')
t = np.linspace(0,2*np.pi,100)
plt.plot(RADIUS_MEASUREMENTS*np.cos(t),RADIUS_MEASUREMENTS*np.sin(t),label='measurement circle')
ax = plt.gca()
ax.set_xlim([-1.4, 1.4])
ax.set_ylim([-1.4, 1.4])
plt.legend()
plt.plot


## Solving the problem

Since we wish to see intermediate iterates, we explicitely instantiate the solver and the stoprule instead of simply calling
sett.run('NewtCG',max_it = 100).
We choose parameters of solver and stopping rule slightly more aggressively than the default values.

In [ ]:
RHO_NEWTON_CG = 0.6 # residual reduction factor per Newton step
TAU = 1.1 # discrepancy principle trigger at first iterate for which the residual is smaller than TAU times noise level

from regpy.stoprules import CountIterations, Discrepancy
from regpy.solvers.nonlinear import NewtonCG

solver = NewtonCG(sett, rho=RHO_NEWTON_CG)
stoprule = Discrepancy(tau=1.1)

for reco,reco_data in solver.until(stoprule=stoprule):
    fig, axs = plt.subplots(1, 2,figsize=(15,7))
    axs[0].set_title(f'Obstacle, it {solver.iteration_step_nr}')
    axs[0].set_xlim([-1.4, 1.4])
    axs[0].set_ylim([-1.4, 1.4])
    axs[1].set_title('Heat flux')

    axs[0].plot(*close_curve(exact_solution.z),label='exact')
    axs[0].plot(*close_curve(op.curve.z),label='reco')
    axs[0].plot(*close_curve(init.z),label='init')    
    axs[0].legend()

    axs[1].plot(exact_data.real, label='exact')
    axs[1].plot(reco_data.real, label='reco')
    axs[1].plot(sett.data.real, '.', label='measured')
    axs[1].legend()
    axs[1].set_ylim(ymin=0)

    plt.show()
